# RF Complexity Analysis (EU)
## Data Loading

We use the `forest_report.json` file in the repo and the unified ETL cache system.

In [1]:
# import os
# os.environ['FORCE_RESULTS_REFRESH'] = '1'

from pathlib import Path
from etl.loader import etl

RESULTS_DIR = Path("results")
zip_paths = sorted(RESULTS_DIR.glob("*.zip"))

# Load ONLY DB10 (LOGS) - shared with prepare_models_analysis to avoid re-reading ZIP files
# Note: verbose=False suppresses dataset selection messages since they're not relevant here
db = etl(
    zip_paths,
    RESULTS_DIR,
    use_cache=True,           # Use the unified cache system
    force_refresh=False,      # Set to True to force refresh
    auto_select=True,         # Auto-select first dataset without prompting
    load_only_db10=True,      # Load ONLY DB10, skip DB0-DB9 to save RAM (16GB+)
    verbose=False             # Suppress selection messages (we load ALL datasets via prepare_models_analysis)
)

# Prepare models analysis - will reuse DB10 from db instead of re-reading ZIPs
# NOTE: processes ALL 25 datasets (selected_dataset=None means "all datasets")
from etl.tables import (
    prepare_models_analysis,
    print_models_analysis_diagnostics,
)

analysis_context = prepare_models_analysis(db=db, verbose=True, selected_dataset=None)

⚠️  WARNING: Skipping corrupted/empty ZIP file: FacesUCR_1_false_0.zip ()
⚠️  WARNING: Skipping corrupted/empty ZIP file: FordA_-1_false_0.zip ()
Report built for 32 workers.
Using unified database (db parameter) for DB10 data sharing
Resuming from cache: 25 datasets already processed
Manifest not found in directory: C:\Users\danie\Projects\GitHub\IEEE_CAI\results\_cache

Processed 0 new datasets (25 total)
Saved redis summary cache at C:\Users\danie\Projects\GitHub\IEEE_CAI\results\redis_reason_counts.csv
Base dir           : C:\Users\danie\Projects\GitHub\IEEE_CAI
Loaded 88 rows from C:\Users\danie\Projects\GitHub\IEEE_CAI\forest_report.json
Results directory  : C:\Users\danie\Projects\GitHub\IEEE_CAI\results (exists=True)


In [2]:
analysis_context.first_table.summary_styler

,dataset,analyzed,train_size,test_size,series_length,n_estimators,mean eu features,eu std,n_features,eu_complexity,eu_min,eu_max
0,Wine,YES,57,54,234,10,3.453,0.710,86.000,297.000,3.000,6.000
1,Wafer,YES,1000,6164,152,10,4.674,1.629,129.000,603.000,3.000,10.000
2,MiddlePhalanxOutlineCorrect,YES,600,291,80,10,18.000,5.725,80.000,1440.000,7.000,39.000
3,MelbournePedestrian,YES,1138,2319,24,10,60.667,10.515,24.000,1456.000,37.000,82.000
4,ChlorineConcentration,YES,467,3840,166,10,10.795,3.718,166.000,1792.000,4.000,25.000
5,ScreenType,NO,375,375,720,10,4.363,1.421,600.000,2618.000,3.000,10.000
6,FordA,YES,3601,1320,500,10,17.622,4.115,500.000,8811.000,7.000,31.000
7,FordB,NO,3636,810,500,10,17.928,4.101,500.000,8964.000,8.000,33.000
8,ElectricDevices,NO,8926,7711,96,10,310.906,61.024,96.000,29847.000,189.000,433.000
9,SonyAIBORobotSurface1,YES,20,601,70,17,3.312,0.583,32.000,106.000,3.000,5.000


In [3]:
print_models_analysis_diagnostics(analysis_context)
analysis_context.summary_styler

? BASE_DIR     : C:\Users\danie\Projects\GitHub\IEEE_CAI
? RESULTS_DIR  : C:\Users\danie\Projects\GitHub\IEEE_CAI\results
Dataset: BeetleFly, BirdChicken, ChlorineConcentration, CinCECGTorso, Coffee, ECG200, ECG5000, FaceFour, FacesUCR, FordA, GunPoint, HandOutlines, ItalyPowerDemand, Lightning2, Meat, MelbournePedestrian, MiddlePhalanxOutlineCorrect, MoteStrain, OliveOil, SonyAIBORobotSurface1, SonyAIBORobotSurface2, ToeSegmentation2, TwoLeadECG, Wafer, Wine


,dataset,Candidate,Reason,Non-reason,Candidate Anti-reason,Anti-reason,Good profile,Bad profile,Preferred reason,Anti-reason profile,Total,Worker start (min),Worker end (max),Worker span (s)
0,BeetleFly,214447.000,9319.000,1695.000,460178.000,2066.000,9319.000,1693.000,0.000,1705.000,700422.000,2025-10-31T17:33:55.030720,2025-11-01T15:14:59.491532,78064.461
19,SonyAIBORobotSurface1,0.000,4902.000,727.000,303484.000,12237.000,3784.000,573.000,0.000,260.000,325967.000,2025-10-25T13:54:38.357814,2025-10-25T15:50:32.599164,6954.241
11,HandOutlines,3607.000,1931.000,411.000,50810.000,555.000,1865.000,349.000,0.000,359.000,59887.000,2025-10-25T16:18:06.781607,2025-10-26T01:18:11.677513,32404.896
6,ECG5000,338578.000,1806.000,0.000,0.000,0.000,32.000,0.000,1778.000,0.000,342194.000,2025-10-27T13:27:28.261700,2025-10-27T20:45:04.251000,26255.989
12,ItalyPowerDemand,0.000,1294.000,357.000,0.000,4942.000,939.000,292.000,0.000,24.000,7848.000,2025-11-05T13:36:49.558941,2025-11-05T15:04:42.836589,5273.278
22,TwoLeadECG,0.000,1153.000,551.000,484385.000,22215.000,845.000,457.000,0.000,398.000,510004.000,2025-11-03T11:33:25.015754,2025-11-03T14:30:46.229607,10641.214
10,GunPoint,0.000,1123.000,316.000,147517.000,6083.000,792.000,239.000,0.000,187.000,156257.000,2025-11-01T15:24:53.803827,2025-11-01T15:44:15.857284,1162.053
1,BirdChicken,152906.000,906.000,1320.000,370182.000,1353.000,906.000,1318.000,0.000,1296.000,530187.000,2025-10-31T11:04:08.989794,2025-10-31T16:55:21.871700,21072.882
9,FordA,0.000,741.000,0.000,0.000,0.000,32.000,0.000,710.000,0.000,1483.000,2025-10-27T03:04:11.473588,2025-10-27T11:42:00.774662,31069.301
17,MoteStrain,155416.000,718.000,8.000,114422.000,39.000,40.000,8.000,698.000,39.000,271388.000,2025-11-04T12:22:31.170654,2025-11-05T12:54:33.144505,88321.974


In [4]:
analysis_context.combined_analyzed_styler

dataset,Wine,Wafer,MiddlePhalanxOutlineCorrect,MelbournePedestrian,ChlorineConcentration,FordA,SonyAIBORobotSurface1,BeetleFly,TwoLeadECG,ECG5000,HandOutlines,Lightning2,FaceFour,FacesUCR,ToeSegmentation2,ECG200,ItalyPowerDemand,Meat,SonyAIBORobotSurface2,Coffee,BirdChicken,GunPoint,CinCECGTorso,OliveOil,MoteStrain
Train Size,57,1000,600,1138,467,3601,20,20,23,500,1000,60,24,200,36,100,67,60,27,28,20,50,40,30,20
Test Size,54,6164,291,2319,3840,1320,601,20,1139,4500,370,61,88,2050,130,100,1029,60,953,28,20,150,1380,30,1252
Series Length,234,152,80,24,166,500,70,512,82,140,2709,637,350,131,343,96,24,448,65,286,512,150,1639,570,84
N Estimators,10,10,10,10,10,10,17,26,54,54,59,65,84,92,98,101,169,193,217,233,233,233,245,284,300
Mean EU Features,3.453,4.674,18.000,60.667,10.795,17.622,3.312,3.071,3.515,5.659,3.252,3.102,3.127,9.377,3.138,4.042,5.500,3.077,3.333,3.111,3.024,3.276,3.102,3.062,3.263
EU Std,0.710,1.629,5.725,10.515,3.718,4.115,0.583,0.258,0.925,2.140,0.558,0.370,0.418,2.661,0.379,1.148,2.082,0.266,0.532,0.314,0.152,0.484,0.354,0.300,0.714
Total Time (ms),580118,35463843,1469590,27536924,17065467,14214037,93066,4947272,33251,21911954,5999882,1482647,15835,34429917,2710865,45937485,4850,30077438,4952624,26156013,4285450,6119,8650001,58648720,2205104
ICF Checks,2164,6174,976,7789,8244,2487,32738,24975,56944,6831,5141,2042,1700,5487,4013,3507,98010,3142,2051,5382,8494,15932,1128,2341,3218
Reason Check Iteration,235916,196066056,228277,223435165,58210694,18140265,37299,124132,20677,117919507,2458537,144482,2901,116640753,1002539,13825002,12849,7017067,2054688,6255421,249253,9676,2032222,147696387,657010
IterGoodRadio %,20.5%,100.0%,89.9%,100.0%,100.0%,100.0%,71.6%,13.3%,52.3%,100.0%,76.3%,83.0%,71.9%,100.0%,88.5%,87.0%,57.5%,96.4%,91.2%,57.5%,5.7%,60.8%,92.5%,100.0%,93.3%
